In [1]:
# Imports
import os
import random
from glob import glob
from PIL import Image
import numpy as np

# PyTorch and Computer Vision libraries
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
# Data Paths and Discovery

cable_dir = r"C:\Users\mkrol\PycharmProjects\ZAW_projects\project01\cable"

# Find all test images
test_images = glob(os.path.join(cable_dir, 'test', '*', '*.png'))
defect_images = [img for img in test_images if 'good' not in img]
good_test_images = [img for img in test_images if 'good' in img]

# Find all training images
train_images = glob(os.path.join(cable_dir, 'train', '*', '*.png'))

In [3]:
# Data Loading and Preprocessing

data_pairs = []
# Match defect images with their corresponding ground truth masks
for img_path in defect_images:
    parts = img_path.split(os.sep)
    defect_type = parts[-2]
    file_name = parts[-1]
    mask_name = file_name.replace('.png', '_mask.png')
    gt_path = os.path.join(cable_dir, 'ground_truth', defect_type, mask_name)

    if os.path.exists(gt_path):
        data_pairs.append({
            'image': img_path,
            'mask': gt_path,
            'is_defective': 1 
        })

test_ds = []
TARGET_SIZE = (256, 256)

# Load defective test images and masks into memory
for item in data_pairs:
    img_path = item['image']
    mask_path = item['mask']
    is_defective = item['is_defective']

    img = Image.open(img_path).convert('RGB').resize(TARGET_SIZE)
    mask = Image.open(mask_path).convert('L').resize(TARGET_SIZE)

    img_array = np.array(img)
    # Convert mask to binary (0 or 255)
    mask_array = (np.array(mask) > 128).astype(np.uint8) * 255

    test_ds.append([img_array, mask_array, is_defective])

print(f'Length of test dataset (defects only): {len(test_ds)}')

# Load good test images into memory 
for path in good_test_images:
    img = Image.open(path).convert('RGB').resize(TARGET_SIZE)
    img_array = np.array(img)
    mask_array = np.zeros(TARGET_SIZE, dtype=np.uint8)
    test_ds.append([img_array, mask_array, 0])

print(f'Total length of test dataset: {len(test_ds)}')

# Load training images into memory
train_ds = []
for path in train_images:
    img = Image.open(path).convert('RGB').resize(TARGET_SIZE)
    img_array = np.array(img)
    mask = np.zeros(TARGET_SIZE, dtype=np.uint8)
    train_ds.append([img_array, mask, 0])

print(f'Length of train dataset: {len(train_ds)}')

Length of test dataset (defects only): 84
Total length of test dataset: 130
Length of train dataset: 224


In [4]:
#Data Split

# Shuffle the test dataset and split it to enrich the training set
random.shuffle(test_ds)
split = int(0.5 * len(test_ds))

# 50% of the test data goes to train, the rest goes to validation
train_data = train_ds + test_ds[:split]
val_data = test_ds[split:]

print(f'Final Training size: {len(train_data)}')
print(f'Final Validation size: {len(val_data)}')

Final Training size: 289
Final Validation size: 65


In [5]:
# Data Augmentation 

# Augmentations for training 
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Transforms for validation 
val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

C:\Users\mkrol\PycharmProjects\ZAW_projects\venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [6]:
# PyTorch Dataset and DataLoaders
class CableDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_array, mask_array, label = self.data[idx]

        if self.transform:
            augmented = self.transform(image=img_array, mask=mask_array)
            img_tensor = augmented['image']
            mask_tensor = augmented['mask']
        else:
            img_tensor = torch.from_numpy(img_array).permute(2, 0, 1).float() / 255.0
            mask_tensor = torch.from_numpy(mask_array).float() / 255.0

        mask_tensor = (mask_tensor > 0.5).float()
        if len(mask_tensor.shape) == 2:
            mask_tensor = mask_tensor.unsqueeze(0)

        return img_tensor, mask_tensor


# Create DataLoaders
train_loader = DataLoader(CableDataset(train_data, transform=train_transform), batch_size=8, shuffle=True)
val_loader = DataLoader(CableDataset(val_data, transform=val_transform), batch_size=1)

In [7]:
# Model Initialization

# Using Unet
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation='sigmoid'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track

In [8]:
# Training Loop

# Dice Loss is highly effective for imbalanced segmentation tasks
criterion = smp.losses.DiceLoss(mode='binary')
optimizer = optim.Adam(model.parameters(), lr=0.0001)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{num_epochs}] completed. Average Loss: {avg_loss:.4f}")

Epoch [1/20] completed. Average Loss: 0.7943
Epoch [2/20] completed. Average Loss: 0.6849
Epoch [3/20] completed. Average Loss: 0.7949
Epoch [4/20] completed. Average Loss: 0.7405
Epoch [5/20] completed. Average Loss: 0.6860
Epoch [6/20] completed. Average Loss: 0.7101
Epoch [7/20] completed. Average Loss: 0.7930
Epoch [8/20] completed. Average Loss: 0.7394
Epoch [9/20] completed. Average Loss: 0.6583
Epoch [10/20] completed. Average Loss: 0.6853
Epoch [11/20] completed. Average Loss: 0.7662
Epoch [12/20] completed. Average Loss: 0.7629
Epoch [13/20] completed. Average Loss: 0.7388
Epoch [14/20] completed. Average Loss: 0.6577
Epoch [15/20] completed. Average Loss: 0.6847
Epoch [16/20] completed. Average Loss: 0.7927
Epoch [17/20] completed. Average Loss: 0.7113
Epoch [18/20] completed. Average Loss: 0.6303
Epoch [19/20] completed. Average Loss: 0.7906
Epoch [20/20] completed. Average Loss: 0.7654


In [9]:
# Evaluation (Intersection over Union)

def calculate_iou(pred, target):
    # Convert probabilities to binary mask (0 or 1)
    pred = (pred > 0.5).float()
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    # If both target and prediction are empty (correctly detected good cable), IoU = 1.0
    if union == 0:
        return 1.0

    return (intersection / (union + 1e-7)).item()


model.eval()
ious = []
with torch.no_grad():
    for images, masks in val_loader:
        images = images.to(device)
        outputs = model(images)
        ious.append(calculate_iou(outputs.cpu(), masks))

print(f"Mean IoU on Validation Set: {np.mean(ious):.4f}")

Mean IoU on Validation Set: 0.4632


In [10]:
# Save the Model
torch.save(model.state_dict(), "cable_model.pth")
print("Model saved successfully as cable_model.pth")

Model saved successfully as cable_model.pth
